## 과제 A: 단위 변환기 (실무 빈도 ★★★★★)

길이·무게·온도 변환 도구. 반환값을 `ConversionResult` Pydantic 객체로.

**힌트**
```python
class ConversionResult(BaseModel):
    original_value: float
    original_unit: str
    converted_value: float
    converted_unit: str

@tool
def convert_length(value: float, from_unit: str, to_unit: str) -> ConversionResult:
    """길이 단위를 변환합니다.

    Args:
        value: 변환할 값
        from_unit: 출발 단위 ('m', 'cm', 'km', 'inch', 'ft')
        to_unit: 도착 단위 ('m', 'cm', 'km', 'inch', 'ft')
    """
    # 변환 로직
    ...
```

테스트: "180cm가 몇 미터야?", "10km는 몇 마일이야?"

---

In [3]:
# 1️⃣ 실행 환경 감지
try:
    import google.colab
    IS_COLAB = True
    print("✅ Google Colab")
except ImportError:
    IS_COLAB = False
    print("✅ 로컬 Jupyter")

✅ 로컬 Jupyter


In [4]:
# 3️⃣ API 키 로드
import os

if IS_COLAB:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
else:
    from dotenv import load_dotenv
    load_dotenv()

print("✅ API 키 OK" if os.environ.get("OPENAI_API_KEY") else "❌ 키 없음")

✅ API 키 OK


In [7]:
# import
# 전역 모듈(라이브러리)
from typing import Literal, Optional
from pydantic import BaseModel, Field   # 1-2교시 정형화 도구
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langchain_core.tools import tool   # ⭐ 오늘의 주인공
    ## tool : 랭체인 프레임워크의 모듈(= 컴포넌트 = 라이브러리) 중 하나
    ## @tool : tool 데코레이터

# Tool Calling은 temperature=0 권장 (도구 선택의 일관성)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

print("✅ import 완료")

✅ import 완료


In [8]:
# 헬퍼 함수
def run_with_tools_for_measure(question: str, tools: list, llm_obj=llm):
    """질문 + 도구 리스트를 받아 4단계 흐름을 실행하고 최종 답변 반환"""
    llm_with_tools = llm_obj.bind_tools(tools)
    messages = [HumanMessage(content=question)]

    response = llm_with_tools.invoke(messages)
    messages.append(response)

    # tool-calling이 필요 없는 경우 (llm이 답변할 수 있는 질문)
    if not response.tool_calls:
        return response.content

    tool_map = {t.name: t for t in tools}
    for tc in response.tool_calls:
        result = tool_map[tc["name"]].invoke(tc["args"])
        messages.append(ToolMessage(content=str(result), tool_call_id=tc["id"]))

    return llm_with_tools.invoke(messages).content

print("✅ 헬퍼 함수 준비 완료")

✅ 헬퍼 함수 준비 완료


In [9]:
# Pydantic 객체로 정의
class ConversionResult(BaseModel):
    original_value: float
    original_unit: str
    converted_value: float
    converted_unit: str

# 기준 단위(m, kg) 대비 배율 테이블
LENGTH_TO_M = {"m": 1, "cm": 0.01, "km": 1000, "inch": 0.0254, "ft": 0.3048, "mile": 1609.344}
WEIGHT_TO_KG = {"kg": 1, "g": 0.001, "lb": 0.45359237, "oz": 0.028349523125}

def convert_by_ratio(value: float, from_unit: str, to_unit: str, table: dict) -> ConversionResult:
    """출발 단위 → 기준 단위 → 도착 단위 순으로 환산"""
    f, t = from_unit.lower(), to_unit.lower()
    for u in (f, t):
        if u not in table:
            raise ValueError(f"지원하지 않는 단위: {u} (지원 단위: {list(table)})")
    converted = value * table[f] / table[t]
    return ConversionResult(
        original_value=value, original_unit=from_unit,
        converted_value=round(converted, 6), converted_unit=to_unit,
    )

# 단위(measure) 함수 정의
@tool
def convert_length(value: float, from_unit: str, to_unit: str) -> ConversionResult:
    """길이 단위를 변환합니다.

    Args:
        value: 변환할 값
        from_unit: 출발 단위 ('m', 'cm', 'km', 'inch', 'ft', 'mile')
        to_unit: 도착 단위 ('m', 'cm', 'km', 'inch', 'ft', 'mile')
    """
    return convert_by_ratio(value, from_unit, to_unit, LENGTH_TO_M)


In [10]:
@tool
def convert_weight(value: float, from_unit: str, to_unit: str) -> ConversionResult:
    """무게 단위를 변환합니다.

    Args:
        value: 변환할 값
        from_unit: 출발 단위 ('kg', 'g', 'lb', 'oz')
        to_unit: 도착 단위 ('kg', 'g', 'lb', 'oz')
    """
    return convert_by_ratio(value, from_unit, to_unit, WEIGHT_TO_KG)


In [11]:
@tool
def convert_temperature(value: float, from_unit: str, to_unit: str) -> ConversionResult:
    """온도 단위를 변환합니다.

    Args:
        value: 변환할 값
        from_unit: 출발 단위 ('C'(섭씨), 'F'(화씨), 'K'(켈빈))
        to_unit: 도착 단위 ('C'(섭씨), 'F'(화씨), 'K'(켈빈))
    """
    # 비례 관계가 아니라서 섭씨를 중간 기준으로 거친다
    to_celsius = {"C": lambda v: v, "F": lambda v: (v - 32) * 5 / 9, "K": lambda v: v - 273.15}
    from_celsius = {"C": lambda c: c, "F": lambda c: c * 9 / 5 + 32, "K": lambda c: c + 273.15}

    f, t = from_unit.upper(), to_unit.upper()
    for u in (f, t):
        if u not in to_celsius:
            raise ValueError(f"지원하지 않는 단위: {u} (지원 단위: {list(to_celsius)})")

    converted = from_celsius[t](to_celsius[f](value))
    return ConversionResult(
        original_value=value, original_unit=from_unit,
        converted_value=round(converted, 6), converted_unit=to_unit,
    )


In [12]:
measure_tools = [convert_length, convert_weight, convert_temperature]

for q in ["180cm가 몇 미터야?", "10km는 몇 마일이야?"]:
    print(f"Q. {q}")
    print(f"A. {run_with_tools_for_measure(q, measure_tools)}\n")


Q. 180cm가 몇 미터야?
A. 180cm는 1.8미터입니다.

Q. 10km는 몇 마일이야?
A. 10킬로미터는 약 6.21마일입니다.



In [13]:
for q in ["몸무게 70kg은 몇 파운드야?", "체온 36.5도(섭씨)는 화씨로 얼마야?", "100℉는 켈빈으로?"]:
    print(f"Q. {q}")
    print(f"A. {run_with_tools_for_measure(q, measure_tools)}\n")


Q. 몸무게 70kg은 몇 파운드야?
A. 70kg은 약 154.32파운드입니다.

Q. 체온 36.5도(섭씨)는 화씨로 얼마야?
A. 체온 36.5도(섭씨)는 화씨로 97.7도입니다.

Q. 100℉는 켈빈으로?
A. 100℉는 약 310.93K입니다.



## 과제 D: docstring 일부러 짧게 써보기 (실험 ★★★★★)

같은 도구의 docstring을 **자세한 버전 vs 짧은 버전**으로 만들어 비교.

```python
@tool
def add_v1(a: int, b: int) -> int:
    """더하기"""        # ← 너무 짧음
    return a + b

@tool
def add_v2(a: int, b: int) -> int:
    """두 정수의 합을 계산합니다.
    수학 계산이 필요한 모든 경우에 사용하세요."""
    return a + b

```

같은 질문에 LLM이 어떤 차이를 보이는지 관찰하세요.

---

In [14]:
@tool
def add_v1(a: int, b: int) -> int:
    """더하기"""        # ← 너무 짧음
    return a + b

@tool
def add_v2(a: int, b: int) -> int:
    """두 정수의 합을 계산합니다.
    수학 계산이 필요한 모든 경우에 사용하세요."""
    return a + b

In [15]:
def check_tool_call(question: str, tools: list):
    """최종 답변 대신 '어떤 도구를 어떤 인자로 골랐는지'만 확인 (미호출이면 LLM 답변 반환)"""
    response = llm.bind_tools(tools).invoke(question)
    return [(tc["name"], tc["args"]) for tc in response.tool_calls] or f"(도구 미호출) {response.content}"

questions = [
    "3 더하기 5는?",
    "사과 12개랑 배 7개 있으면 과일은 모두 몇 개야?",
    "123456 + 654321 계산해줘",
    "오늘 서울 날씨 어때?",
]

for q in questions:
    print(f"Q. {q}")
    print("  v1 (짧은 docstring)  :", check_tool_call(q, [add_v1]))
    print("  v2 (자세한 docstring):", check_tool_call(q, [add_v2]))
    print("  v1 + v2 동시 제공    :", check_tool_call(q, [add_v1, add_v2]))
    print()


Q. 3 더하기 5는?
  v1 (짧은 docstring)  : [('add_v1', {'a': 3, 'b': 5})]
  v2 (자세한 docstring): [('add_v2', {'a': 3, 'b': 5})]
  v1 + v2 동시 제공    : [('add_v1', {'a': 3, 'b': 5})]

Q. 사과 12개랑 배 7개 있으면 과일은 모두 몇 개야?
  v1 (짧은 docstring)  : [('add_v1', {'a': 12, 'b': 7})]
  v2 (자세한 docstring): [('add_v2', {'a': 12, 'b': 7})]
  v1 + v2 동시 제공    : [('add_v1', {'a': 12, 'b': 7})]

Q. 123456 + 654321 계산해줘
  v1 (짧은 docstring)  : [('add_v1', {'a': 123456, 'b': 654321})]
  v2 (자세한 docstring): [('add_v2', {'a': 123456, 'b': 654321})]
  v1 + v2 동시 제공    : [('add_v1', {'a': 123456, 'b': 654321}), ('add_v2', {'a': 123456, 'b': 654321})]

Q. 오늘 서울 날씨 어때?
  v1 (짧은 docstring)  : (도구 미호출) 죄송하지만, 현재 날씨 정보를 제공할 수 있는 기능이 없습니다. 서울의 날씨를 확인하려면 기상청 웹사이트나 날씨 앱을 이용해 주세요.
  v2 (자세한 docstring): (도구 미호출) 죄송하지만, 현재 날씨 정보를 제공할 수 없습니다. 서울의 날씨를 확인하려면 기상청 웹사이트나 날씨 앱을 이용해 주세요.
  v1 + v2 동시 제공    : (도구 미호출) 죄송하지만, 현재 날씨 정보를 제공할 수 없습니다. 서울의 날씨를 확인하려면 기상청 웹사이트나 날씨 앱을 이용해 주세요.

